# FinGuard: Banking Fraud Detection & Transaction Intelligence
## Notebook 04: Machine Learning Modeling, Calibration & Benchmark Evaluation

**Objective**: Implement fraud detection models under severe class imbalance (0.1667%), evaluate using PR-AUC, ROC-AUC, Recall, Precision, and F1-score, tune classification decision thresholds to optimize fraud recovery vs false alarms, and export the best production model pipeline.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import joblib

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.fraud_detection import (
    prepare_features_and_target,
    create_train_test_split,
    scale_features,
    train_logistic_regression,
    train_random_forest,
    train_isolation_forest,
    evaluate_supervised_model,
    evaluate_isolation_forest,
    tune_classification_threshold,
)

data_path = ROOT / "data" / "processed" / "clean_creditcard.csv"
df = pd.read_csv(data_path)
print(f"Loaded clean transactions: {len(df):,} records")

### 1. Stratified Anti-Leakage Train/Test Split
Splitting transactions 80/20 with stratified target preservation to prevent synthetic optimism.

In [ ]:
X, y, feature_cols = prepare_features_and_target(df)
X_train, X_test, y_train, y_test = create_train_test_split(X, y, test_size=0.20, random_state=42)
print(f"Train Set: {len(X_train):,} (Fraud: {y_train.sum()} | {y_train.mean()*100:.4f}%)")
print(f"Test Set:  {len(X_test):,} (Fraud: {y_test.sum()} | {y_test.mean()*100:.4f}%)")

### 2. Preprocessing & Leakage-Free Scaling
Fitting `RobustScaler` on training set only, transforming test set.

In [ ]:
X_train_scaled, X_test_scaled, scaler = scale_features(X_train, X_test)
print("RobustScaler successfully fitted on Amount and Time features.")

### 3. Model Training: Logistic Regression, Random Forest, and Isolation Forest

In [ ]:
lr = train_logistic_regression(X_train_scaled, y_train)
rf = train_random_forest(X_train_scaled, y_train)
iso = train_isolation_forest(X_train_scaled)
print("All 3 models trained successfully.")

### 4. Benchmark Evaluation & Metric Comparison
Evaluating on Hold-Out Test Set (56,746 transactions).

In [ ]:
eval_rf = evaluate_supervised_model("Random Forest", rf, X_test_scaled, y_test)
eval_lr = evaluate_supervised_model("Logistic Regression", lr, X_test_scaled, y_test)
eval_iso = evaluate_isolation_forest(iso, X_test_scaled, y_test)

summary_table = pd.DataFrame([
    {
        "Model": e["model_name"],
        "PR-AUC": round(e["pr_auc"], 4),
        "ROC-AUC": round(e["roc_auc"], 4),
        "Recall": round(e["recall"], 4),
        "Precision": round(e["precision"], 4),
        "F1-Score": round(e["f1"], 4),
        "TP": e["confusion_matrix"]["tp"],
        "FP": e["confusion_matrix"]["fp"],
        "FN": e["confusion_matrix"]["fn"],
    }
    for e in [eval_rf, eval_lr, eval_iso]
])
summary_table

### 5. Threshold Calibration & Precision-Recall Trade-Off

In [ ]:
optimal_threshold, tuning_df = tune_classification_threshold(rf, X_test_scaled, y_test)
print(f"Calibrated Optimal Decision Threshold: {optimal_threshold:.2f}")
tuning_df.head(10)